In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!apt-get install -y fonts-nanum

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-nanum is already the newest version (20200506-1).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.


"""
역사 인근 편의점 시간대별 재고 및 진열 최적화 분석
──────────────────────────────────────────────────
피라미드 근거 통계분석: OLS 단순회귀분석 + 잔차 진단
히트맵 시각화: Claus Wilke 원칙 적용 (비례잉크·의미있는색상·데이터기반정렬)

데이터:
- Seoul_subway_data_20210705.csv (역별 시간대별 승하차인원)
- subway_location_data.csv (역별 위경도 좌표)
"""

In [ ]:
import glob, os
base = '/content/drive/MyDrive'
subway_files = glob.glob(base + '/**/Seoul_subway_data_20210705*.csv', recursive=True)
loc_files = glob.glob(base + '/**/subway_location_data*.csv', recursive=True)
print('SUBWAY:', subway_files)
print('LOC:', loc_files)

SUBWAY: ['/content/drive/MyDrive/이어드림스쿨(공유)/dataset/Seoul_subway_data_20210705 (1).csv']
LOC: ['/content/drive/MyDrive/이어드림스쿨(공유)/dataset/subway_location_data (1).csv']


In [ ]:
import pandas as pd, numpy as np
from scipy import stats
import statsmodels.api as sm
df = pd.read_csv(subway_files[0], encoding='cp949', on_bad_lines='skip')
df_loc = pd.read_csv(loc_files[0], encoding='utf-8-sig')
df_loc['지하철역_키'] = df_loc['지하철역'].str.replace('역$', '', regex=True)
df['오전_출근_하차'] = df['07시-08시 하차인원'] + df['08시-09시 하차인원']
df['오후_퇴근_승차'] = df['18시-19시 승차인원'] + df['19시-20시 승차인원']
mid_hours = ['10시-11시','11시-12시','12시-13시','13시-14시','14시-15시','15시-16시']
mid_cols = [col for col in df.columns if any(h in col for h in mid_hours)]
df['통제_낮시간대_총이용객'] = df[mid_cols].sum(axis=1)
station_summary = df.groupby('지하철역')[['오전_출근_하차','오후_퇴근_승차','통제_낮시간대_총이용객']].mean().reset_index()
m_df = pd.merge(station_summary, df_loc, left_on='지하철역', right_on='지하철역_키', how='inner')
x = m_df['오전_출근_하차'].values
y = m_df['오후_퇴근_승차'].values
c = m_df['통제_낮시간대_총이용객'].values
n = len(x)
slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
r_simple = r_value**2
X_multi = sm.add_constant(np.column_stack([x, c]))
model = sm.OLS(y, X_multi).fit()
r_multi = model.rsquared
beta_x = model.params[1]
p_x = model.pvalues[1]
beta_c = model.params[2]
p_c = model.pvalues[2]
corr_xc = np.corrcoef(x, c)[0,1]
residuals_multi = y - model.predict(X_multi)
m_df['잔차_통제'] = residuals_multi
m_df['유형'] = m_df['잔차_통제'].apply(lambda r: '퇴근집중형' if r > 0 else '출근집중형')
n_evening = int((m_df['유형']=='퇴근집중형').sum())
n_morning = int((m_df['유형']=='출근집중형').sum())
print('=' * 60)
print('STEP: 통제변수(낮시간대 총이용객) 추가 재분석 결과 - 상관관계 검증')
print('=' * 60)
print('분석 대상:', n, '개 역')
print('[단순회귀] R2=', round(r_simple,4), ' slope=', round(slope,4), ' p=', p_value)
print('[통제회귀] R2=', round(r_multi,4))
print(' 베타(오전하차 X, 통제후):', round(beta_x,4), ' p=', p_x)
print(' 베타(낮시간대 통제변수):', round(beta_c,4), ' p=', p_c)
print(' X와 통제변수의 상관계수:', round(corr_xc,4))
print('역 유형(통제모델 잔차 기준): 퇴근집중형', n_evening, '개, 출근집중형', n_morning, '개')

STEP: 통제변수(낮시간대 총이용객) 추가 재분석 결과 - 상관관계 검증
분석 대상: 570 개 역
[단순회귀] R2= 0.9422  slope= 0.9907  p= 0.0
[통제회귀] R2= 0.9815
 베타(오전하차 X, 통제후): 0.7469  p= 2.54031866e-316
 베타(낮시간대 통제변수): 0.119  p= 1.595078658739838e-142
 X와 통제변수의 상관계수: 0.7693
역 유형(통제모델 잔차 기준): 퇴근집중형 303 개, 출근집중형 267 개


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.font_manager as fm
FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
BOLD_PATH = '/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf'
KR = fm.FontProperties(fname=FONT_PATH)
KR_B = fm.FontProperties(fname=BOLD_PATH)
fig = plt.figure(figsize=(18,11))
fig.patch.set_facecolor('white')
fig.text(0.5, 0.96, '오전 하차 vs 오후 승차 상관관계 분석 (통제변수 반영)', fontproperties=KR_B, fontsize=15, ha='center')
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.28, top=0.9, bottom=0.08)
ax1 = fig.add_subplot(gs[0,0])
colors = ['#2549D8' if r > 0 else '#F4A261' for r in residuals_multi]
ax1.scatter(x, y, c=colors, alpha=0.5, s=22)
xs = np.linspace(x.min(), x.max(), 200)
ax1.plot(xs, slope*xs+intercept, color='#E63946', lw=2)
ax1.set_title('단순회귀(통제 전) R2=' + str(round(r_simple,3)), fontproperties=KR_B, fontsize=11)
ax1.set_xlabel('오전 출근 하차인원', fontproperties=KR, fontsize=9)
ax1.set_ylabel('오후 퇴근 승차인원', fontproperties=KR, fontsize=9)
ax2 = fig.add_subplot(gs[0,1])
ax2.scatter(c, y, alpha=0.35, s=18, color='#457B9D')
slope_cy, intercept_cy, r_cy, p_cy, se_cy = stats.linregress(c, y)
xs2 = np.linspace(c.min(), c.max(), 200)
ax2.plot(xs2, slope_cy*xs2+intercept_cy, color='#E63946', lw=2)
ax2.set_title('통제변수(낮시간대 총이용객) vs Y R2=' + str(round(r_cy**2,3)), fontproperties=KR_B, fontsize=11)
ax2.set_xlabel('낮시간대(10-16시) 총이용객', fontproperties=KR, fontsize=9)
ax3 = fig.add_subplot(gs[1,0])
ax3.axis('off')
rows = [['단순회귀 R2', str(round(r_simple,4))], ['통제회귀 R2(X+통제변수)', str(round(r_multi,4))], ['X 계수(통제후)', str(round(beta_x,4))], ['X p-value(통제후)', format(p_x,'.2e')], ['통제변수 계수', str(round(beta_c,4))], ['통제변수 p-value', format(p_c,'.2e')], ['X-통제변수 상관계수', str(round(corr_xc,4))]]
tbl = ax3.table(cellText=rows, colLabels=['통계량','값'], loc='center', cellLoc='center')
for (r_i, c_i), cell in tbl.get_celld().items():
    cell.get_text().set_fontproperties(KR_B if r_i == 0 else KR)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1,1.6)
ax4 = fig.add_subplot(gs[1,1])
ax4.scatter(model.predict(X_multi), residuals_multi, alpha=0.4, s=18, color='#457B9D')
ax4.axhline(0, color='#E63946', ls='--')
ax4.set_title('통제모델 잔차 vs 적합값', fontproperties=KR_B, fontsize=11)
out_dir = loc_files[0].rsplit('/',1)[0] + '/'
plt.savefig(out_dir+'fig1_regression_analysis_v2.png', dpi=130, bbox_inches='tight', facecolor='white')
plt.show()
print('fig1_regression_analysis_v2.png 저장 완료:', out_dir+'fig1_regression_analysis_v2.png')

fig1_regression_analysis_v2.png 저장 완료: /content/drive/MyDrive/이어드림스쿨(공유)/dataset/fig1_regression_analysis_v2.png


In [ ]:
hour_bins = ['04시-05시','05시-06시','06시-07시','07시-08시','08시-09시','09시-10시','10시-11시','11시-12시','12시-13시','13시-14시','14시-15시','15시-16시','16시-17시','17시-18시','18시-19시','19시-20시','20시-21시','21시-22시','22시-23시','23시-24시','00시-01시','01시-02시','02시-03시','03시-04시']
for h in hour_bins:
    df[h+'_합계'] = df[h+' 승차인원'] + df[h+' 하차인원']
station_col = '지하철역_x' if '지하철역_x' in m_df.columns else ('지하철역' if '지하철역' in m_df.columns else '지하철역_y')
station_type_map = m_df.drop_duplicates(subset=[station_col]).set_index(station_col)['유형']
df['유형'] = df['지하철역'].map(station_type_map)
df_typed = df.dropna(subset=['유형'])
hourly_cols = [h+'_합계' for h in hour_bins]
type_hourly = df_typed.groupby('유형')[hourly_cols].mean()
type_hourly = type_hourly.reindex(['퇴근집중형','출근집중형'])
type_hourly_norm = type_hourly.div(type_hourly.sum(axis=1), axis=0) * 100
fig2, ax = plt.subplots(figsize=(16,4.2))
im = ax.imshow(type_hourly_norm.values, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(hour_bins)))
ax.set_xticklabels(hour_bins, fontproperties=KR, fontsize=8, rotation=45, ha='right')
ax.set_yticks(range(len(type_hourly_norm.index)))
ax.set_yticklabels(type_hourly_norm.index, fontproperties=KR_B, fontsize=11)
for yi in range(type_hourly_norm.shape[0]):
    for xi in range(type_hourly_norm.shape[1]):
        val = type_hourly_norm.values[yi, xi]
        ax.text(xi, yi, f'{val:.1f}', ha='center', va='center', fontsize=6.5, color='black' if val < type_hourly_norm.values.max()*0.6 else 'white')
ax.set_title('역 유형별(통제모델 잔차 기준 분류) 시간대별 이용량 비중(%) - 상관관계 기반 서술적 비교', fontproperties=KR_B, fontsize=12)
cbar = fig2.colorbar(im, ax=ax, fraction=0.02, pad=0.01)
cbar.set_label('일평균 이용량 대비 비중(%)', fontproperties=KR, fontsize=8)
plt.tight_layout()
fig2.savefig(out_dir+'fig2_hourly_pattern_heatmap.png', dpi=130, bbox_inches='tight', facecolor='white')
plt.close(fig2)
print('fig2_hourly_pattern_heatmap.png 저장 완료:', out_dir+'fig2_hourly_pattern_heatmap.png')
from IPython.display import Image, display
display(Image(out_dir+'fig2_hourly_pattern_heatmap.png'))

fig2_hourly_pattern_heatmap.png 저장 완료: /content/drive/MyDrive/이어드림스쿨(공유)/dataset/fig2_hourly_pattern_heatmap.png
